<a href="https://colab.research.google.com/github/SaiDurga98/Machine-Learning/blob/main/Building_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANN architecture:
i/p layer -> 784 nodes (784 features)
hidden layer 1 -> 128 neurons -> reLU
hidden layer 2 -> 64 features -> reLU
output -> 10 neurons -> softwax since its a multi class classification problem


# Workflow:
1. Create dataloader objects for both taining and test data
2. Write training loop
3. Evaluate model using test data

In [30]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim
import matplotlib.pyplot as plt

In [31]:
# sets the starting point for Pytorch random number generator
# so the outcome is the same every time you run your code
torch.manual_seed(42)

In [32]:
# check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [33]:
# About datset:Each row is one image of a clothing item (28 *28 = 784 pixels grayscale)
# The 2D image is flattened into a single row so pixel one through pixel 784 are the columns
# Plus the label column = 785 total columns . The label is what you're predicting : 0-9 each number being a clothing type
# The pixel values are 0-255 where 0 is black and 255 is white. The most values in the data are 0 because edges of each image are just black and actual clothing shape sits in the middle.
# This 28 * 28 is theat will be feeded to our neural network i.e.. the input layer will need 784 neurons - one per pixel
#df = pd.read_csv('fmnist_small.csv') # just small dataset with few samples
df = pd.read_csv('fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [34]:
df.shape

(60000, 785)

In [35]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [37]:
# Scaling the features
# Dividing by 255(their max pixel value) squashes every pixel from the 0-255 range down to 0-1.
# This is called normalization. Neural n/w learns better with small numbers.
#Training works by multiplying inputs with weights and nudging those weights via gradients. If inputs are big (like 255), the multiplications produce large values, gradients can swing wildly, and training becomes unstable or slow — like trying to parallel park a car whose steering wheel is way too sensitive. With inputs between 0 and 1, the updates are smooth and controlled.

X_train = X_train/255.0
X_test = X_test/255.0

In [38]:
# Create CustomDataset class
class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype = torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [39]:
# Create train_dataset object
train_dataset = CustomDataset(X_train, y_train)

In [40]:
len(train_dataset)

48000

In [ ]:
train_dataset[0]

In [41]:
test_dataset = CustomDataset(X_test, y_test)

In [42]:
len(test_dataset)

12000

In [43]:
# create train and test dataLoader objects
# pin_memory=True is a speed optimization for when you're training on a GPU.
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)


In [44]:
# Define Neural network class
class MyNeuralNetwork(nn.Module):

  def __init__(self, features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(features, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128, 64), #output of first layer goes into next
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64, 10)
        # pytorch handles softmax implicitly at output layer so not need to apply softmax
    )

  def forward(self, x):
    return self.model(x)



In [45]:
# Set learning rate and epochs
epochs = 100
learning_rate = 0.1

In [46]:
# Instantiate model
model = MyNeuralNetwork(X_train.shape[1])
# Move model to GPU
model.to(device)
# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer # weight_decay is regularization coefficients
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [47]:
# Training Loop

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_dataloader:
    # Move data to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = criterion(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_dataloader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 0.6249084657629331
Epoch: 2 , Loss: 0.49199690653880435
Epoch: 3 , Loss: 0.45562089485426743
Epoch: 4 , Loss: 0.43380642544229825
Epoch: 5 , Loss: 0.41715061584611735
Epoch: 6 , Loss: 0.40564093277355034
Epoch: 7 , Loss: 0.3941608931571245
Epoch: 8 , Loss: 0.38580174928406874
Epoch: 9 , Loss: 0.3743983890265226
Epoch: 10 , Loss: 0.3725726637095213
Epoch: 11 , Loss: 0.36783315147956214
Epoch: 12 , Loss: 0.3572052289446195
Epoch: 13 , Loss: 0.35052060889204345
Epoch: 14 , Loss: 0.3449219484726588
Epoch: 15 , Loss: 0.34472562207778296
Epoch: 16 , Loss: 0.33732124184072015
Epoch: 17 , Loss: 0.3344038988550504
Epoch: 18 , Loss: 0.3302020480086406
Epoch: 19 , Loss: 0.33063985937833784
Epoch: 20 , Loss: 0.3262277270356814
Epoch: 21 , Loss: 0.3208496819138527
Epoch: 22 , Loss: 0.3183093272894621
Epoch: 23 , Loss: 0.3225850373158852
Epoch: 24 , Loss: 0.31459670132398604
Epoch: 25 , Loss: 0.31343053522954384
Epoch: 26 , Loss: 0.31424527982374034
Epoch: 27 , Loss: 0.3107087447295

In [49]:
# Set model to eval mode
model.eval()

MyNeuralNetwork(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [51]:
# Evaluation code
total = 0
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in test_dataloader:
     # Move data to GPU
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    # Apply forward pass for prediction values
    outputs = model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total += batch_labels.shape[0] # (32+32+32...)
    correct += (predicted == batch_labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy: {accuracy}')



Accuracy: 89.10833333333333


# **How the accuracy above is calculated:**

 You send in 1 batch of 32 test images → the model returns a tensor of shape (32, 10) — one row per image, 10 scores per row (one for each clothing class). Each row's highest-score position is the predicted label.

So the key idea: the model doesn't answer "7" directly. It answers with 10 raw scores (called logits), and you pick the winner — usually with torch.max(outputs, 1) or argmax, which returns the index of the largest score in each row. That index is the predicted class.

That's exactly what your evaluation loop will do next: compare those 32 argmax picks against the 32 true labels, count matches, and that's your accuracy.

# Optimizing neural network:
**Overfitting** occurs when model's accuracy is good on training data but not that accurate on test data.

Optimization techniques:

1.Regularization
2.Dropout
3.Batch Normalization

# Batch Normalization
1. Applied after linear layer and before activation function

Your input data is nicely scaled between 0 and 1 — but only at the front door. After the data passes through the first layer (multiplied by weights, summed, ReLU'd), the numbers coming out can be all over the place — some huge, some tiny. And it gets worse: as training updates the weights of layer 1, the distribution of values flowing into layer 2 keeps shifting every single step. Layer 2 is trying to learn from an input that keeps changing its scale and center — like trying to hit a moving target. This shifting is often called internal covariate shift.

Batch norm fixes it by re-standardizing the activations at each layer, for each batch: take the batch's values at that layer, subtract the mean, divide by the standard deviation — so they're re-centered around 0 with a consistent spread — then let the network scale/shift them via two small learnable parameters (gamma and beta) if it prefers something other than exactly 0-mean-1-std.